# 04b Predict Candidate Pairs

Run the configured LLM or Jev/System 1 predictions for the active year slice. Inputs are loaded from saved candidate and abstract files, and each prediction backend resumes from its own store. This notebook does not build the screened dataset.

## Inputs

- `candidate_amr_pair_year_counts.parquet`, filtered to the configured year slice.
- `pmid_abstracts.parquet` prepared by notebook 04a.
- Active source, model, run ID, and resume settings from `outputs/pipeline_config.yaml`.
- Existing LLM or System 1 prediction store for resume; only the selected backend is used.

## Outputs

- Updated `yearly_amr_unified_decisions.parquet` when the LLM source is active, or `yearly_amr_system1_decisions.parquet` when System 1 is active. The stores remain separate and retain prior results.


In [1]:
%load_ext autoreload
%autoreload 2


In [2]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "amr").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "amr").exists():
    raise RuntimeError(f"Could not find repository root from {Path.cwd()}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from amr.pipeline_config import cfg, cfg_path, load_pipeline_config

from amr.screening import (
    LLM_CLASSIFICATION_MAX_TOKENS,
    UNIFIED_AMR_INTEREST_SYSTEM_PROMPT,
    prepare_yearly_candidate_pmids,
    run_unified_llm_filter,
    run_system1_filter,
    OpenRouterJevClient,
)
from amr.llm import OpenAIConfig, OpenAILanguageModel


In [3]:
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "amr").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
PIPELINE_CONFIG = load_pipeline_config(project_root=PROJECT_ROOT)

ASSETS_DIR = cfg_path(PIPELINE_CONFIG, "paths.assets_dir", PROJECT_ROOT)
OUTPUT_DIR = cfg_path(PIPELINE_CONFIG, "paths.output_dir", PROJECT_ROOT)
PARQUET_OUTPUT_DIR = cfg_path(PIPELINE_CONFIG, "paths.parquet_dir", PROJECT_ROOT)
PARQUET_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = PARQUET_OUTPUT_DIR

PATHOGENS_PATH = cfg_path(PIPELINE_CONFIG, "paths.pathogens", PROJECT_ROOT)
PATHOGEN_PRIORITY_PATH = cfg_path(PIPELINE_CONFIG, "paths.pathogen_priority", PROJECT_ROOT)
MIN_PATHOGENIC_INDEX = cfg(PIPELINE_CONFIG, "collection.min_pathogenic_index")
ANTIBIOTIC_COARSENING_PATH = cfg_path(PIPELINE_CONFIG, "paths.antibiotic_coarsening", PROJECT_ROOT)
ANTIBIOTIC_SYNONYM_PATH = cfg_path(PIPELINE_CONFIG, "paths.antibiotic_synonyms", PROJECT_ROOT)
OUTPUT_YEARLY_AMR_PATH = cfg_path(PIPELINE_CONFIG, "paths.yearly_candidates", PROJECT_ROOT)
OUTPUT_YEARLY_SCREEN_PATH = cfg_path(PIPELINE_CONFIG, "paths.yearly_screened", PROJECT_ROOT)
OUTPUT_UNIFIED_DECISIONS_PATH = cfg_path(PIPELINE_CONFIG, "paths.unified_decisions", PROJECT_ROOT)
OUTPUT_SYSTEM1_DECISIONS_PATH = cfg_path(PIPELINE_CONFIG, "paths.system1_decisions", PROJECT_ROOT)
OUTPUT_PATHOGEN_BG_PATH = cfg_path(PIPELINE_CONFIG, "paths.pathogen_background", PROJECT_ROOT)
OUTPUT_ANTIBIOTIC_BG_PATH = cfg_path(PIPELINE_CONFIG, "paths.antibiotic_background", PROJECT_ROOT)
INTERVAL_PMIDS_PATH = cfg_path(PIPELINE_CONFIG, "paths.interval_pmids", PROJECT_ROOT)
YEARLY_PAIR_PMID_PATH = cfg_path(PIPELINE_CONFIG, "paths.yearly_pair_pmids", PROJECT_ROOT)
PUBMED_METADATA_CACHE_PATH = cfg_path(PIPELINE_CONFIG, "paths.pubmed_metadata_cache", PROJECT_ROOT)
ABSTRACT_CACHE_PATH = cfg_path(PIPELINE_CONFIG, "paths.abstract_cache", PROJECT_ROOT)
PUBMED_EMAIL = cfg(PIPELINE_CONFIG, "collection.pubmed_email")

# Date windows: TOTAL_* defines the canonical range; SLICE_* limits this run for partitioned/resumable work.
TOTAL_START_YEAR = cfg(PIPELINE_CONFIG, "collection.total_start_year")
TOTAL_END_YEAR = cfg(PIPELINE_CONFIG, "collection.total_end_year")
TOTAL_MIN_DATE = f"{TOTAL_START_YEAR}/01/01"
TOTAL_MAX_DATE = f"{TOTAL_END_YEAR}/12/31"
SLICE_START_YEAR = cfg(PIPELINE_CONFIG, "collection.slice_start_year")
SLICE_END_YEAR = cfg(PIPELINE_CONFIG, "collection.slice_end_year")
MIN_DATE = f"{SLICE_START_YEAR}/01/01"
MAX_DATE = f"{SLICE_END_YEAR}/12/31"
INTERVAL_YEAR_GAP = cfg(PIPELINE_CONFIG, "collection.interval_year_gap")

# Vocabulary limits: use None for full canonical runs; integers are useful for quick/resumable subsets.
NUM_PATHOGENS = cfg(PIPELINE_CONFIG, "collection.num_pathogens")
NUM_ANTIBIOTICS = cfg(PIPELINE_CONFIG, "collection.num_antibiotics")
EXTRA_AMR_KEYWORDS = cfg(PIPELINE_CONFIG, "collection.extra_amr_keywords", [])

# Select which saved prediction source builds the screened dataset.
ACTIVE_PREDICTION_SOURCE = str(cfg(PIPELINE_CONFIG, "screening.active_source", "llm")).lower()
SYSTEM1_PROBABILITY_THRESHOLD = float(cfg(PIPELINE_CONFIG, "screening.system1_probability_threshold", 0.5))
SYSTEM1_MODEL = cfg(PIPELINE_CONFIG, "system1.model", "typesafe/jev-1.13")
SYSTEM1_RUN_ID = str(cfg(PIPELINE_CONFIG, "system1.run_id", "jev-default"))
SYSTEM1_RUN_FILTER = bool(cfg(PIPELINE_CONFIG, "system1.run_filter", True))
SYSTEM1_MAX_CALLS = cfg(PIPELINE_CONFIG, "system1.max_calls")
SYSTEM1_RATE_LIMIT = float(cfg(PIPELINE_CONFIG, "system1.rate_limit", 0.0))
SYSTEM1_MAX_WORKERS = int(cfg(PIPELINE_CONFIG, "system1.max_workers", 32))
SYSTEM1_SAVE_EVERY = int(cfg(PIPELINE_CONFIG, "system1.save_every", 100))
SYSTEM1_PROGRESS_EVERY = int(cfg(PIPELINE_CONFIG, "system1.progress_every", 100))
if ACTIVE_PREDICTION_SOURCE not in {"llm", "system1"}:
    raise ValueError("screening.active_source must be 'llm' or 'system1'.")
if not 0.0 <= SYSTEM1_PROBABILITY_THRESHOLD <= 1.0:
    raise ValueError("screening.system1_probability_threshold must be between 0 and 1.")

# LLM decision modes: rerun_unified_decisions=True ignores active decisions but reuses cached abstracts.
RUN_LLM_FILTER = cfg(PIPELINE_CONFIG, "llm.run_filter")
RERUN_UNIFIED_LLM_DECISIONS = cfg(PIPELINE_CONFIG, "llm.rerun_unified_decisions")
MAX_LLM_CALLS = cfg(PIPELINE_CONFIG, "llm.max_calls")
UNIFIED_SYSTEM_PROMPT = UNIFIED_AMR_INTEREST_SYSTEM_PROMPT
LLM_MAX_WORKERS = cfg(PIPELINE_CONFIG, "llm.max_workers")
LLM_BATCH_SIZE = cfg(PIPELINE_CONFIG, "llm.batch_size")
FAILED_RETRY_PATH = cfg_path(PIPELINE_CONFIG, "paths.unified_failures", PROJECT_ROOT)
MAX_FAILURE_RETRIES = cfg(PIPELINE_CONFIG, "llm.max_failure_retries")
ABSTRACT_FETCH_BATCH_SIZE = cfg(PIPELINE_CONFIG, "llm.abstract_fetch_batch_size")
PREFETCH_ABSTRACTS = cfg(PIPELINE_CONFIG, "llm.prefetch_abstracts")
LLM_CATEGORY_BACKFILL_DECISIONS = cfg(PIPELINE_CONFIG, "llm.category_backfill_decisions")
LLM_RETRY_CACHED_FAILURES = bool(cfg(PIPELINE_CONFIG, "llm.retry_cached_failures", False))
LLM_INCLUDE_EXPLANATORY_TEXT = bool(cfg(PIPELINE_CONFIG, "llm.include_explanatory_text", False))
LLM_REASONING_EFFORT = cfg(PIPELINE_CONFIG, "llm.reasoning_effort", "minimal")
UNIFIED_PROGRESS_EVERY = cfg(PIPELINE_CONFIG, "llm.progress_every")
SCREENED_INCLUDE_DECISIONS = tuple(cfg(PIPELINE_CONFIG, "llm.screened_include_decisions", ["strict", "yes"]))

# PubMed throughput and background-count controls.
COLLECTION_VERBOSE = cfg(PIPELINE_CONFIG, "collection.collection_verbose")
COLLECTION_PROGRESS_EVERY = cfg(PIPELINE_CONFIG, "collection.collection_progress_every")
PAIR_FULL_FETCH_THRESHOLD = cfg(PIPELINE_CONFIG, "collection.pair_full_fetch_threshold")
COLLECTION_SAVE_EVERY = cfg(PIPELINE_CONFIG, "collection.save_every", 500)
COLLECTION_SUMMARY_BATCH_SIZE = cfg(PIPELINE_CONFIG, "collection.summary_batch_size", 1000)
COLLECTION_RETRIEVE_BATCH_SIZE = cfg(PIPELINE_CONFIG, "collection.retrieve_batch_size", 9000)
COLLECTION_MAX_WORKERS = cfg(PIPELINE_CONFIG, "collection.max_workers", 4)
COLLECTION_MAX_PENDING_FUTURES = cfg(PIPELINE_CONFIG, "collection.max_pending_futures", 8)
COLLECTION_PARTIAL_INTERVAL_MAX = cfg(PIPELINE_CONFIG, "collection.partial_interval_max", 1)
BACKGROUND_SCOPE = cfg(PIPELINE_CONFIG, "collection.background_scope")
BACKGROUND_FETCH_THRESHOLD = cfg(PIPELINE_CONFIG, "collection.background_fetch_threshold")
BACKGROUND_METADATA_BATCH_SIZE = cfg(PIPELINE_CONFIG, "collection.background_metadata_batch_size")


## Load the saved candidate set and run predictions

In [4]:
if not OUTPUT_YEARLY_AMR_PATH.exists():
    raise FileNotFoundError(
        f"Missing candidate data at {OUTPUT_YEARLY_AMR_PATH}. Run notebook 04 collection first."
    )
yearly_amr_df = pd.read_parquet(OUTPUT_YEARLY_AMR_PATH)
print(f"Loaded yearly candidate rows from disk: {len(yearly_amr_df):,}")

candidate_df = prepare_yearly_candidate_pmids(yearly_amr_df)
candidate_df = candidate_df.loc[
    candidate_df["year"].between(SLICE_START_YEAR, SLICE_END_YEAR)
].reset_index(drop=True)
print(f"Yearly candidate PMIDs in slice [{SLICE_START_YEAR}, {SLICE_END_YEAR}]: {len(candidate_df):,}")

unified_df = pd.DataFrame()
system1_df = pd.DataFrame()
if ACTIVE_PREDICTION_SOURCE == "llm":
    resume_df = None
    auto_resume_unified = not RERUN_UNIFIED_LLM_DECISIONS
    category_backfill_decisions = LLM_CATEGORY_BACKFILL_DECISIONS
    active_failed_retry_path = FAILED_RETRY_PATH
    if RERUN_UNIFIED_LLM_DECISIONS:
        active_failed_retry_path = PARQUET_OUTPUT_DIR / "yearly_amr_unified_failures_prompt_rerun.parquet"
        category_backfill_decisions = None
        print(
            "RERUN_UNIFIED_LLM_DECISIONS=True: rerunning LLM classification. "
            f"Cached abstracts at {ABSTRACT_CACHE_PATH} will still be reused."
        )
    elif OUTPUT_UNIFIED_DECISIONS_PATH.exists():
        resume_df = pd.read_parquet(OUTPUT_UNIFIED_DECISIONS_PATH)
        retry_msg = " and cached LLM failures" if LLM_RETRY_CACHED_FAILURES else ""
        print(
            f"Found existing unified decisions: {len(resume_df):,} rows. "
            f"Missing candidates{retry_msg} will be processed automatically."
        )

    if RUN_LLM_FILTER:
        llm = OpenAILanguageModel(OpenAIConfig(model=cfg(PIPELINE_CONFIG, "llm.model"), max_tokens=LLM_CLASSIFICATION_MAX_TOKENS, temperature=0))
        unified_df = run_unified_llm_filter(
            candidate_df,
            llm,
            email=PUBMED_EMAIL,
            system_prompt=UNIFIED_SYSTEM_PROMPT,
            output_path=str(OUTPUT_UNIFIED_DECISIONS_PATH),
            resume_from=resume_df,
            auto_resume_from_output=auto_resume_unified,
            llm_max_workers=LLM_MAX_WORKERS,
            llm_batch_size=LLM_BATCH_SIZE,
            abstract_fetch_batch_size=ABSTRACT_FETCH_BATCH_SIZE,
            prefetch_abstracts=PREFETCH_ABSTRACTS,
            failed_retry_path=str(active_failed_retry_path),
            max_failure_retries=MAX_FAILURE_RETRIES,
            retry_cached_llm_failures=LLM_RETRY_CACHED_FAILURES,
            abstract_cache_path=str(ABSTRACT_CACHE_PATH),
            max_llm_calls=MAX_LLM_CALLS,
            category_backfill_decisions=category_backfill_decisions,
            include_explanatory_text=LLM_INCLUDE_EXPLANATORY_TEXT,
            reasoning_effort=LLM_REASONING_EFFORT,
            progress_every=UNIFIED_PROGRESS_EVERY,
            verbose=1,
        )
    elif resume_df is not None and not resume_df.empty:
        unified_df = resume_df
    elif OUTPUT_UNIFIED_DECISIONS_PATH.exists():
        unified_df = pd.read_parquet(OUTPUT_UNIFIED_DECISIONS_PATH)
    else:
        raise FileNotFoundError(
            f"No unified decisions file found at {OUTPUT_UNIFIED_DECISIONS_PATH}. "
            "Enable llm.run_filter to create it."
        )
    print(unified_df["unified_decision"].value_counts(dropna=False))
else:
    if SYSTEM1_RUN_FILTER:
        jev = OpenRouterJevClient(model=SYSTEM1_MODEL)
        system1_df = run_system1_filter(
            candidate_df,
            jev,
            model=SYSTEM1_MODEL,
            run_id=SYSTEM1_RUN_ID,
            output_path=OUTPUT_SYSTEM1_DECISIONS_PATH,
            abstract_cache_path=ABSTRACT_CACHE_PATH,
            email=PUBMED_EMAIL,
            save_every=SYSTEM1_SAVE_EVERY,
            progress_every=SYSTEM1_PROGRESS_EVERY,
            rate_limit=SYSTEM1_RATE_LIMIT,
            max_workers=SYSTEM1_MAX_WORKERS,
            max_calls=SYSTEM1_MAX_CALLS,
        )
    elif OUTPUT_SYSTEM1_DECISIONS_PATH.exists():
        system1_df = pd.read_parquet(OUTPUT_SYSTEM1_DECISIONS_PATH)
    else:
        raise FileNotFoundError(
            f"No System 1 decisions file found at {OUTPUT_SYSTEM1_DECISIONS_PATH}. "
            "Enable system1.run_filter to create it."
        )
    if "system1_run_id" in system1_df.columns:
        system1_df = system1_df.loc[system1_df["system1_run_id"].astype(str) == SYSTEM1_RUN_ID]
    print(system1_df["system1_status"].value_counts(dropna=False))
    print(
        f"System 1 run {SYSTEM1_RUN_ID}: {len(system1_df):,} predictions; "
        f"P(yes) threshold for screening: {SYSTEM1_PROBABILITY_THRESHOLD:.2f}"
    )


Loaded yearly candidate rows from disk: 41,531
Yearly candidate PMIDs in slice [2000, 2026]: 186,724
System 1 run jev-1.13-amr-support-v1: 186,724 candidate rows; resuming 55,000 completed; 131,724 pending this session; up to 128 concurrent requests.
55,100/186,724 candidates (29.5%) | this session 100/131,724 | successes 54,889 | request/parse failures 0 | abstract failures 0 | no abstract 211 | API attempts this session 100 (100 succeeded, 0 failed) | elapsed 0m 00s | ETA this session 16m 37s | account credits remaining $8.1363 | end balance estimate waiting for OpenRouter to report spend
55,200/186,724 candidates (29.6%) | this session 200/131,724 | successes 54,989 | request/parse failures 0 | abstract failures 0 | no abstract 211 | API attempts this session 200 (200 succeeded, 0 failed) | elapsed 0m 01s | ETA this session 12m 21s | account credits remaining $8.1363 | end balance estimate waiting for OpenRouter to report spend
55,300/186,724 candidates (29.6%) | this session 300/13